<a href="https://colab.research.google.com/github/Anugya43/virtual-fashion-advisor/blob/main/fashion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import ast  # converts string → dictionary

df = pd.read_csv("filtered_fashion_data.csv")

# Convert attribute string into dictionary
df["attribute_dict"] = df["p_attributes"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else {})

# Extract Occasion
df["occasion"] = df["attribute_dict"].apply(lambda x: x.get("Occasion", None))

# Drop rows without occasion
df = df.dropna(subset=["occasion"])

# Check distribution
print(df["occasion"].value_counts())

occasion
Festive    100
Daily      100
Sports     100
Party      100
Fusion      61
Formal      59
Work        41
Western     39
Name: count, dtype: int64


In [ ]:
def map_occasion(x):
    if x in ["Casual", "Daily"]:
        return "Casual"
    elif x in ["Festive", "Traditional", "Ethnic"]:
        return "Festive"
    elif x in ["Party"]:
        return "Party"
    elif x in ["Formal", "Work"]:
        return "Formal"
    elif x in ["Sports"]:
        return "Sports"
    elif x in ["Western", "Fusion"]:
        return "Western"
    else:
        return None

df["occasion_clean"] = df["occasion"].apply(map_occasion)

df = df.dropna(subset=["occasion_clean"])

In [ ]:
print(df["occasion_clean"].value_counts())

occasion_clean
Festive    100
Western    100
Casual     100
Formal     100
Party      100
Sports     100
Name: count, dtype: int64


In [ ]:
df_balanced = df.groupby("occasion_clean").head(100)

In [ ]:
df_balanced.to_csv("filtered_fashion_data.csv", index=False)


In [ ]:
valid_ids = set(df_balanced["id"].astype(str))

KeyError: 'id'

In [ ]:
import pandas as pd
import ast

df = pd.read_csv("filtered_fashion_data.csv")

# Convert attributes
df["attr_dict"] = df["p_attributes"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else {})

# Extract occasion
df["occasion"] = df["attr_dict"].apply(lambda x: x.get("Occasion", None))

# Remove junk
df = df.dropna(subset=["occasion", "img"])
df = df[df["occasion"] != "NA"]

# Keep only required columns
df = df[["p_id", "name", "colour", "img", "occasion"]]

In [ ]:
df["image_file"] = df["p_id"].astype(str) + ".jpg"

In [ ]:
import os

image_folder = "images"

df = df[df["image_file"].apply(lambda x: os.path.exists(os.path.join(image_folder, x)))]

In [ ]:
df = df.groupby("occasion").head(100)

In [ ]:
import shutil

target_folder = "filtered_images"
os.makedirs(target_folder, exist_ok=True)

for img in df["image_file"]:
    shutil.copy(
        os.path.join(image_folder, img),
        os.path.join(target_folder, img)
    )

In [ ]:
df.to_csv("final_dataset.csv", index=False)

In [ ]:
df.to_json("data.json", orient="records")

In [2]:
import pandas as pd
import ast



df = pd.read_csv("filtered_fashion_data.csv")

# Remove unnamed columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]


def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except:
        return {}

df["attribute_dict"] = df["attribute_dict"].apply(safe_eval)




important_features = [
    "Top Shape",
    "Top Length",
    "Top Pattern",
    "Print or Pattern Type",
    "Neck",
    "Sleeve Length",
    "Top Hemline",
    "Occasion",
    "Main Trend",
    "Top Fabric",
]

for feature in important_features:

    col_name = feature.lower().replace(" ", "_")

    df[col_name] = df["attribute_dict"].apply(
        lambda x: x.get(feature, None)
    )


# PRINT UNIQUE VALUES


for feature in important_features:

    col_name = feature.lower().replace(" ", "_")

    print("\n" + "="*50)
    print(f"FEATURE: {feature}")
    print("="*50)

    values = (
        df[col_name]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .value_counts()
    )

    print(values.head(50))


FEATURE: Top Shape
top_shape
straight    81
a-line      49
anarkali    28
kaftan       3
pathani      1
Name: count, dtype: int64

FEATURE: Top Length
top_length
calf length     104
above knee       32
knee length      15
floor length     11
Name: count, dtype: int64

FEATURE: Top Pattern
top_pattern
printed         87
embroidered     28
solid           18
yoke design     15
self design      8
striped          4
woven design     1
checked          1
Name: count, dtype: int64

FEATURE: Print or Pattern Type
print_or_pattern_type
solid                 244
ethnic motifs         110
floral                 94
geometric              27
striped                23
abstract               17
woven design           12
embellished            11
colourblocked          10
bandhani                8
polka dots              6
checked                 6
self design             6
paisley                 4
typography              4
textured                3
leheriya                2
graphic                

In [4]:
import pandas as pd
import ast
import numpy as np

# LOAD DATA


df = pd.read_csv("filtered_fashion_data.csv")

# Remove unnamed columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# CLEAN ATTRIBUTE COLUMN

def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except:
        return {}

df["attribute_dict"] = df["attribute_dict"].apply(safe_eval)

# =========================
# EXTRACT IMPORTANT FIELDS
# =========================

features = [
    "Top Shape",
    "Neck",
    "Top Hemline",
    "Print or Pattern Type",
    "Occasion",
]

for feature in features:

    col = feature.lower().replace(" ", "_")

    df[col] = df["attribute_dict"].apply(
        lambda x: x.get(feature, np.nan)
    )

# =========================
# CLEAN TEXT
# =========================

cols_to_clean = [
    "top_shape",
    "neck",
    "top_hemline",
    "print_or_pattern_type",
    "occasion_clean",
    "colour"
]

for col in cols_to_clean:

    df[col] = (
        df[col]
        .astype(str)
        .str.lower()
        .str.strip()
    )

# =========================
# BODY TYPE INFERENCE
# =========================

def infer_body_type(row):

    shape = str(row["top_shape"])
    neck = str(row["neck"])
    hemline = str(row["top_hemline"])

    # PEAR
    if shape in ["anarkali", "a-line"]:
        return "pear"

    # APPLE
    elif neck in ["v-neck", "keyhole neck"]:
        return "apple"

    # RECTANGLE
    elif shape == "straight":
        return "rectangle"

    # HOURGLASS
    elif hemline == "flared":
        return "hourglass"

    else:
        return "neutral"

df["body_type"] = df.apply(infer_body_type, axis=1)

# =========================
# UNDERTONE INFERENCE
# =========================

warm_colors = [
    "yellow", "mustard", "orange",
    "brown", "gold", "beige",
    "cream", "olive", "rust"
]

cool_colors = [
    "blue", "purple", "pink",
    "black", "grey", "silver",
    "navy"
]

neutral_colors = [
    "white", "green", "maroon"
]

def infer_undertone(color):

    color = str(color).lower()

    for c in warm_colors:
        if c in color:
            return "warm"

    for c in cool_colors:
        if c in color:
            return "cool"

    for c in neutral_colors:
        if c in color:
            return "neutral"

    return "neutral"

df["undertone"] = df["colour"].apply(infer_undertone)

# =========================
# COUNTS
# =========================

print("\n===== OCCASION COUNTS =====")
print(df["occasion_clean"].value_counts())

print("\n===== BODY TYPE COUNTS =====")
print(df["body_type"].value_counts())

print("\n===== UNDERTONE COUNTS =====")
print(df["undertone"].value_counts())

print("\n===== TOP SHAPE COUNTS =====")
print(df["top_shape"].value_counts())

# =========================
# COMBINED ANALYTICS
# =========================

combo_counts = (
    df.groupby([
        "occasion_clean",
        "undertone",
        "body_type"
    ])
    .size()
    .reset_index(name="count")
)

# Sort highest counts first
combo_counts = combo_counts.sort_values(
    by="count",
    ascending=False
)

print("\n===== COMBINED ANALYTICS =====")
print(combo_counts.head(50))

# =========================
# SAVE ANALYTICS
# =========================

combo_counts.to_csv(
    "fashion_analytics.csv",
    index=False
)

print("\nAnalytics saved successfully.")


===== OCCASION COUNTS =====
occasion_clean
festive    100
western    100
casual     100
formal     100
party      100
sports     100
Name: count, dtype: int64

===== BODY TYPE COUNTS =====
body_type
neutral      405
pear          77
rectangle     67
apple         51
Name: count, dtype: int64

===== UNDERTONE COUNTS =====
undertone
cool       278
neutral    224
warm        98
Name: count, dtype: int64

===== TOP SHAPE COUNTS =====
top_shape
nan         438
straight     81
a-line       49
anarkali     28
kaftan        3
pathani       1
Name: count, dtype: int64

===== COMBINED ANALYTICS =====
   occasion_clean undertone  body_type  count
34         sports      cool    neutral     56
25         formal      cool    neutral     42
30          party      cool    neutral     41
32          party   neutral    neutral     36
35         sports   neutral    neutral     32
27         formal   neutral    neutral     27
38        western      cool    neutral     26
28         formal      warm    ne

In [6]:
import pandas as pd
import ast

# =========================
# LOAD DATASET
# =========================

df = pd.read_csv("filtered_fashion_data.csv")

# Remove unnamed columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# =========================
# CONVERT ATTRIBUTE COLUMN
# =========================

def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except:
        return {}

df["attribute_dict"] = df["attribute_dict"].apply(safe_eval)

# =========================
# GET ALL UNIQUE ATTRIBUTE KEYS
# =========================

all_keys = set()

for row in df["attribute_dict"]:
    all_keys.update(row.keys())

print("\n===== ALL UNIQUE ATTRIBUTE TYPES =====\n")

for key in sorted(all_keys):
    print(key)

# =========================
# GET UNIQUE VALUES FOR EACH ATTRIBUTE
# =========================

for key in sorted(all_keys):

    print("\n" + "="*60)
    print(f"ATTRIBUTE: {key}")
    print("="*60)

    values = []

    for row in df["attribute_dict"]:

        if key in row:

            value = str(row[key]).strip().lower()

            if value != "na":
                values.append(value)

    unique_values = pd.Series(values).value_counts()

    print(unique_values.head(50))


===== ALL UNIQUE ATTRIBUTE TYPES =====

About the Brand
Add-Ons
Better Cotton Initiative
Blouse
Blouse Fabric
Body Shape ID
Body or Garment Size
Border
Bottom Closure
Bottom Fabric
Bottom Pattern
Bottom Type
Brand Fit Name
Care for me
Center Front Open
Character
Closure
Collection Name
Colour Family
Contact Brand or Retailer for pre-sales product queries
Design Styling
Dupatta
Dupatta Border
Dupatta Fabric
Dupatta Pattern
Fabric
Fabric 2
Fabric Purity
Fabric Type
Features
Fit
Fly Type
Fusion Wear
Hemline
Knit or Woven
Kurta Fabric
Kurta Pattern
Length
Lining
Main Trend
Multipack Set
Neck
Number of Pockets
Occasion
Ornamentation
Pattern
Pattern Coverage
Print or Pattern Type
Saree Fabric
Set Content
Shape
Sleeve Length
Sleeve Styling
Slit Detail
Sport
Sport Team
Stitch
Style Tip
Sub Trend
Surface Styling
Sustainable
Technique
Technology
Top Design Styling
Top Fabric
Top Hemline
Top Length
Top Pattern
Top Shape
Top Type
Transparency
Trends
Type
Type of Pleat
Waist Rise
Waistband
Wash Ca

In [7]:
import pandas as pd
import ast

# =========================
# LOAD DATASET
# =========================

df = pd.read_csv("filtered_fashion_data.csv")

# =========================
# CONVERT attribute_dict
# STRING -> REAL DICTIONARY
# =========================

def convert_to_dict(x):
    try:
        return ast.literal_eval(x)
    except:
        return {}

df['attribute_dict'] = df['attribute_dict'].apply(convert_to_dict)

# =========================
# FEATURES WE WANT
# =========================

features = [
    'Top Shape',
    'Top Length',
    'Top Pattern',
    'Print or Pattern Type',
    'Neck',
    'Sleeve Length',
    'Top Hemline',
    'Main Trend',
    'Top Fabric',
    'Colour Family',
    'Fit',
    'Pattern Coverage',
    'Occasion',
    'Type',
    'Waist Rise',
    'Body Shape ID',
    'Where-to-wear',
    'Style Tip',
    'Fabric',
    'Pattern'
]

# =========================
# CREATE NEW COLUMNS
# =========================

for feature in features:

    clean_name = (
        feature.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    df[clean_name] = df['attribute_dict'].apply(
        lambda x: x.get(feature, None)
    )

# =========================
# KEEP IMPORTANT COLUMNS
# =========================

final_columns = [
    'p_id',
    'name',
    'colour',
    'brand',
    'img',
    'description',
    'occasion_clean',

    # extracted columns
    'top_shape',
    'top_length',
    'top_pattern',
    'print_or_pattern_type',
    'neck',
    'sleeve_length',
    'top_hemline',
    'main_trend',
    'top_fabric',
    'colour_family',
    'fit',
    'pattern_coverage',
    'occasion',
    'type',
    'waist_rise',
    'body_shape_id',
    'where_to_wear',
    'style_tip',
    'fabric',
    'pattern'
]

# keep only available columns
final_columns = [col for col in final_columns if col in df.columns]

df = df[final_columns]

# =========================
# SAVE CLEAN DATASET
# =========================

df.to_csv("fashion_extracted.csv", index=False)

print("DONE")
print(df.head())

DONE
       p_id                                               name     colour  \
0  17048614  Khushal K Women Black Ethnic Motifs Printed Ku...      Black   
1  16524740  InWeave Women Orange Solid Kurta with Palazzos...     Orange   
2  16331376  Anubhutee Women Navy Blue Ethnic Motifs Embroi...  Navy Blue   
3  14709966  Nayo Women Red Floral Printed Kurta With Trous...        Red   
4  11056154   AHIKA Women Black & Green Printed Straight Kurta      Black   

       brand                                                img  \
0  Khushal K  http://assets.myntassets.com/assets/images/170...   
1    InWeave  http://assets.myntassets.com/assets/images/165...   
2  Anubhutee  http://assets.myntassets.com/assets/images/163...   
3       Nayo  http://assets.myntassets.com/assets/images/147...   
4      AHIKA  http://assets.myntassets.com/assets/images/110...   

                                         description occasion_clean top_shape  \
0  Black printed Kurta with Palazzos with dupatt

In [9]:

import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("fashion_extracted.csv")

# -------------------------------
# CLEAN TEXT
# -------------------------------

text_cols = [
    "colour",
    "top_shape",
    "neck",
    "top_pattern",
    "pattern_coverage",
    "fit",
    "fabric"
]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower()

# -------------------------------
# UNDERTONE MATCH
# -------------------------------

warm_colors = [
    "orange", "mustard", "yellow", "olive",
    "maroon", "cream", "gold", "brown"
]

cool_colors = [
    "blue", "purple", "pink", "lavender",
    "silver", "navy", "black"
]

neutral_colors = [
    "white", "grey", "beige", "green"
]

def detect_undertone(colour):
    if pd.isna(colour):
        return "neutral"

    colour = str(colour).lower()

    for c in warm_colors:
        if c in colour:
            return "warm"

    for c in cool_colors:
        if c in colour:
            return "cool"

    for c in neutral_colors:
        if c in colour:
            return "neutral"

    return "neutral"

df["undertone_match"] = df["colour"].apply(detect_undertone)

# -------------------------------
# SILHOUETTE TYPE
# -------------------------------

def silhouette(shape):
    if "anarkali" in shape:
        return "flowy"

    elif "a-line" in shape:
        return "balanced"

    elif "straight" in shape:
        return "structured"

    elif "kaftan" in shape:
        return "loose"

    return "neutral"

df["silhouette"] = df["top_shape"].apply(silhouette)

# -------------------------------
# VISUAL WEIGHT
# -------------------------------

def visual_weight(pattern):
    if "large" in pattern:
        return "heavy"

    elif "small" in pattern:
        return "light"

    elif "placement" in pattern:
        return "balanced"

    return "medium"

df["visual_weight"] = df["pattern_coverage"].apply(visual_weight)

# -------------------------------
# BODY BALANCE LOGIC
# -------------------------------

def body_balance(fit):
    if "flared" in fit:
        return "adds volume"

    elif "skinny" in fit:
        return "slimming"

    elif "straight" in fit:
        return "balanced"

    elif "loose" in fit:
        return "concealing"

    return "neutral"

df["body_balance"] = df["fit"].apply(body_balance)

# -------------------------------
# FABRIC STRUCTURE
# -------------------------------

def fabric_structure(fabric):
    if "cotton" in fabric:
        return "structured"

    elif "silk" in fabric:
        return "luxury flow"

    elif "georgette" in fabric:
        return "soft flow"

    elif "polyester" in fabric:
        return "stiff"

    return "regular"

df["fabric_structure"] = df["fabric"].apply(fabric_structure)

# -------------------------------
# SAVE NEW DATASET
# -------------------------------

df.to_csv("fashion_ai_ready_dataset.csv", index=False)

print("AI-ready fashion dataset created successfully.")

AI-ready fashion dataset created successfully.


In [11]:
import pandas as pd
import numpy as np

# =========================================
# LOAD DATASET
# =========================================

df = pd.read_csv("fashion_ai_ready_dataset.csv")

# =========================================
# CLEAN TEXT COLUMNS
# =========================================

cols = [
    "top_shape",
    "neck",
    "fit",
    "pattern_coverage",
    "top_pattern",
    "colour",
    "occasion_clean",
    "fabric",
    "type"
]

for col in cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.lower()
            .str.strip()
        )

# =========================================
# BODY TYPE RULE ENGINE
# =========================================

def assign_body_type(row):

    score = {
        "pear": 0,
        "apple": 0,
        "rectangle": 0,
        "hourglass": 0,
        "inverted_triangle": 0
    }

    top_shape = str(row.get("top_shape", ""))
    neck = str(row.get("neck", ""))
    fit = str(row.get("fit", ""))
    pattern = str(row.get("pattern_coverage", ""))
    sleeve = str(row.get("sleeve_length", ""))
    fabric = str(row.get("fabric", ""))

    # =====================================
    # PEAR BODY RULES
    # =====================================

    if "a-line" in top_shape:
        score["pear"] += 3

    if "boat neck" in neck:
        score["pear"] += 2

    if "flared" in fit:
        score["pear"] += 2

    if "structured" in fabric:
        score["pear"] += 1

    # =====================================
    # APPLE BODY RULES
    # =====================================

    if "v-neck" in neck:
        score["apple"] += 3

    if "straight" in top_shape:
        score["apple"] += 2

    if "vertical" in pattern:
        score["apple"] += 2

    if "long sleeves" in sleeve:
        score["apple"] += 1

    # =====================================
    # RECTANGLE BODY RULES
    # =====================================

    if "layered" in fit:
        score["rectangle"] += 3

    if "anarkali" in top_shape:
        score["rectangle"] += 2

    if "flared" in fit:
        score["rectangle"] += 2

    if "embellished" in pattern:
        score["rectangle"] += 1

    # =====================================
    # HOURGLASS BODY RULES
    # =====================================

    if "fitted" in fit:
        score["hourglass"] += 3

    if "balanced" in pattern:
        score["hourglass"] += 2

    if "v-neck" in neck:
        score["hourglass"] += 1

    if "wrap" in top_shape:
        score["hourglass"] += 2

    # =====================================
    # INVERTED TRIANGLE RULES
    # =====================================

    if "a-line" in top_shape:
        score["inverted_triangle"] += 2

    if "flared" in fit:
        score["inverted_triangle"] += 2

    if "deep neck" in neck:
        score["inverted_triangle"] += 1

    # =====================================
    # FINAL BODY TYPE
    # =====================================

    max_score = max(score.values())

    if max_score == 0:
        return "neutral"

    best_matches = [
        body for body, val in score.items()
        if val == max_score
    ]

    return ",".join(best_matches)

# =========================================
# APPLY BODY TYPE LOGIC
# =========================================

df["body_type"] = df.apply(assign_body_type, axis=1)

# =========================================
# UNDERTONE DETECTION
# =========================================

warm_colors = [
    "orange", "mustard", "yellow",
    "olive", "maroon", "brown",
    "cream", "gold"
]

cool_colors = [
    "blue", "purple", "pink",
    "lavender", "black", "silver",
    "navy"
]

neutral_colors = [
    "white", "grey", "beige",
    "green"
]

def assign_undertone(colour):

    colour = str(colour).lower()

    for c in warm_colors:
        if c in colour:
            return "warm"

    for c in cool_colors:
        if c in colour:
            return "cool"

    for c in neutral_colors:
        if c in colour:
            return "neutral"

    return "neutral"

df["undertone"] = df["colour"].apply(assign_undertone)

# =========================================
# ANALYTICS
# =========================================

print("\n===== BODY TYPE COUNTS =====")
print(df["body_type"].value_counts())

print("\n===== UNDERTONE COUNTS =====")
print(df["undertone"].value_counts())

print("\n===== OCCASION + BODY TYPE =====")
print(
    df.groupby(
        ["occasion_clean", "body_type"]
    ).size().reset_index(name="count")
)

# =========================================
# SAVE FINAL DATASET
# =========================================

df.to_csv("fashion_stylist_dataset.csv", index=False)

print("\nDataset saved successfully.")


===== BODY TYPE COUNTS =====
body_type
neutral                             394
apple                               109
pear                                 47
rectangle                            20
pear,rectangle,inverted_triangle     14
pear,apple                           13
pear,rectangle                        3
Name: count, dtype: int64

===== UNDERTONE COUNTS =====
undertone
cool       265
neutral    246
warm        89
Name: count, dtype: int64

===== OCCASION + BODY TYPE =====
   occasion_clean                         body_type  count
0          casual                             apple     30
1          casual                           neutral     42
2          casual                              pear     14
3          casual                        pear,apple      8
4          casual                    pear,rectangle      3
5          casual                         rectangle      3
6         festive                             apple     42
7         festive                    

In [12]:
import pandas as pd
import ast
import re

# =========================
# LOAD DATASET
# =========================
df = pd.read_csv("fashion_ai_ready_dataset.csv")

# =========================
# SAFE DICT PARSER
# =========================
def parse_dict(x):
    try:
        return ast.literal_eval(x) if pd.notna(x) else {}
    except:
        return {}

df["attribute_dict"] = df["attribute_dict"].apply(parse_dict)

# =========================
# EXTRACT IMPORTANT FEATURES
# =========================

def get_attr(x, key):
    return x.get(key, None) if isinstance(x, dict) else None

features = {
    "top_shape": "Top Shape",
    "top_length": "Top Length",
    "top_pattern": "Top Pattern",
    "print_type": "Print or Pattern Type",
    "neck": "Neck",
    "sleeve_length": "Sleeve Length",
    "top_hemline": "Top Hemline",
    "main_trend": "Main Trend",
    "top_fabric": "Top Fabric",
    "fit": "Fit",
    "pattern_coverage": "Pattern Coverage",
    "occasion": "Occasion",
    "type": "Type",
    "waist_rise": "Waist Rise",
    "where_to_wear": "Where-to-wear",
    "style_tip": "Style Tip",
    "fabric": "Fabric",
    "pattern": "Pattern",
}

for col, key in features.items():
    df[col] = df["attribute_dict"].apply(lambda x: get_attr(x, key))

# =========================
# CLEAN TEXT
# =========================
text_cols = [
    "top_shape", "top_length", "top_pattern",
    "print_type", "neck", "sleeve_length",
    "top_hemline", "main_trend", "top_fabric",
    "fit", "pattern_coverage", "occasion",
    "type", "waist_rise", "where_to_wear",
    "style_tip", "fabric", "pattern",
    "colour"
]

for col in text_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.lower()
        .str.strip()
        .replace("none", pd.NA)
        .replace("nan", pd.NA)
    )

# =========================
# BODY TYPE LABELING
# =========================
# Humans really looked at clothing silhouettes for centuries
# and now we are turning it into if-else statements. Civilization. 💀

def assign_body_type(row):

    shape = str(row["top_shape"])
    hem = str(row["top_hemline"])
    neck = str(row["neck"])
    fit = str(row["fit"])
    length = str(row["top_length"])
    pattern = str(row["print_type"])

    body_types = []

    # PEAR
    if (
        "a-line" in shape
        or "anarkali" in shape
        or "flared" in hem
    ):
        body_types.append("pear")

    # APPLE
    if (
        "straight" in shape
        or "v-neck" in neck
        or "empire" in str(row["main_trend"])
    ):
        body_types.append("apple")

    # RECTANGLE
    if (
        "straight fit" in fit
        or "regular fit" in fit
        or "solid" in pattern
    ):
        body_types.append("rectangle")

    # INVERTED TRIANGLE
    if (
        "boat neck" in neck
        or "off-shoulder" in neck
        or "shoulder straps" in neck
    ):
        body_types.append("inverted_triangle")

    # HOURGLASS
    if (
        "wrap" in str(row["type"])
        or "belt" in str(row["style_tip"])
        or "cinched" in str(row["type"])
    ):
        body_types.append("hourglass")

    if len(body_types) == 0:
        return "neutral"

    return ",".join(sorted(set(body_types)))

df["body_type"] = df.apply(assign_body_type, axis=1)

# =========================
# UNDERTONE LABELING
# =========================

warm_colors = [
    "yellow", "mustard", "orange", "gold",
    "olive", "peach", "maroon", "brown",
    "rust", "cream"
]

cool_colors = [
    "blue", "purple", "pink", "lavender",
    "grey", "gray", "black", "navy",
    "silver", "white"
]

neutral_colors = [
    "green", "red", "beige", "multicolor"
]

def assign_undertone(color):

    if pd.isna(color):
        return "neutral"

    color = str(color).lower()

    for c in warm_colors:
        if c in color:
            return "warm"

    for c in cool_colors:
        if c in color:
            return "cool"

    for c in neutral_colors:
        if c in color:
            return "neutral"

    return "neutral"

df["undertone"] = df["colour"].apply(assign_undertone)

# =========================
# CLEAN DESCRIPTIONS
# =========================

def clean_description(text):

    if pd.isna(text):
        return ""

    text = re.sub(r"<.*?>", " ", str(text))
    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["clean_description"] = df["description"].apply(clean_description)

# =========================
# IMAGE CHECK
# =========================

df = df[df["img"].notna()]
df = df[df["img"] != ""]

# =========================
# ANALYTICS
# =========================

print("\n===== BODY TYPE COUNTS =====")
print(df["body_type"].value_counts())

print("\n===== UNDERTONE COUNTS =====")
print(df["undertone"].value_counts())

print("\n===== OCCASION COUNTS =====")
print(df["occasion_clean"].value_counts())

print("\n===== TOP SHAPE COUNTS =====")
print(df["top_shape"].value_counts())

print("\n===== COMBINED ANALYTICS =====")
combo = (
    df.groupby(
        ["occasion_clean", "undertone", "body_type"]
    )
    .size()
    .reset_index(name="count")
    .sort_values(by="count", ascending=False)
)

print(combo.head(50))

# =========================
# SAVE FINAL DATASET
# =========================

important_cols = [
    "p_id",
    "name",
    "brand",
    "img",
    "colour",
    "occasion_clean",
    "clean_description",
    "top_shape",
    "neck",
    "top_pattern",
    "top_fabric",
    "fit",
    "body_type",
    "undertone"
]

final_df = df[important_cols]

final_df.to_csv(
    "fashion_dataset_processed.csv",
    index=False
)

print("\nDataset saved successfully.")

KeyError: 'attribute_dict'

In [13]:
import pandas as pd
import ast
import re

# ==========================================
# LOAD DATASET
# ==========================================

df = pd.read_csv("fashion_ai_ready_dataset.csv")

print("\n===== DATASET COLUMNS =====")
print(df.columns)

# ==========================================
# DETECT ATTRIBUTE COLUMN
# ==========================================

if "attribute_dict" in df.columns:
    attr_col = "attribute_dict"

elif "p_attributes" in df.columns:
    attr_col = "p_attributes"

else:
    raise Exception(
        "No attribute column found! Expected 'attribute_dict' or 'p_attributes'"
    )

# ==========================================
# SAFE DICT PARSER
# ==========================================

def parse_dict(x):

    try:
        return ast.literal_eval(x) if pd.notna(x) else {}

    except:
        return {}

df[attr_col] = df[attr_col].apply(parse_dict)

# ==========================================
# EXTRACT IMPORTANT FEATURES
# ==========================================

def get_attr(x, key):

    if isinstance(x, dict):
        return x.get(key, None)

    return None

features = {
    "top_shape": "Top Shape",
    "top_length": "Top Length",
    "top_pattern": "Top Pattern",
    "print_type": "Print or Pattern Type",
    "neck": "Neck",
    "sleeve_length": "Sleeve Length",
    "top_hemline": "Top Hemline",
    "main_trend": "Main Trend",
    "top_fabric": "Top Fabric",
    "fit": "Fit",
    "pattern_coverage": "Pattern Coverage",
    "occasion": "Occasion",
    "type": "Type",
    "waist_rise": "Waist Rise",
    "where_to_wear": "Where-to-wear",
    "style_tip": "Style Tip",
    "fabric": "Fabric",
    "pattern": "Pattern",
}

for col, key in features.items():

    df[col] = df[attr_col].apply(
        lambda x: get_attr(x, key)
    )

# ==========================================
# CLEAN TEXT
# ==========================================

text_cols = [
    "top_shape",
    "top_length",
    "top_pattern",
    "print_type",
    "neck",
    "sleeve_length",
    "top_hemline",
    "main_trend",
    "top_fabric",
    "fit",
    "pattern_coverage",
    "occasion",
    "type",
    "waist_rise",
    "where_to_wear",
    "style_tip",
    "fabric",
    "pattern",
    "colour",
]

for col in text_cols:

    if col in df.columns:

        df[col] = (
            df[col]
            .astype(str)
            .str.lower()
            .str.strip()
            .replace("none", pd.NA)
            .replace("nan", pd.NA)
        )

# ==========================================
# CLEAN DESCRIPTION
# ==========================================

def clean_description(text):

    if pd.isna(text):
        return ""

    text = re.sub(r"<.*?>", " ", str(text))
    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["clean_description"] = df["description"].apply(
    clean_description
)

# ==========================================
# BODY TYPE LABELING
# ==========================================
# Fashion stylists: years of expertise
# Us: if neck contains "boat" then inverted triangle 💀

def assign_body_type(row):

    shape = str(row["top_shape"])
    hem = str(row["top_hemline"])
    neck = str(row["neck"])
    fit = str(row["fit"])
    pattern = str(row["print_type"])
    outfit_type = str(row["type"])
    style_tip = str(row["style_tip"])

    body_types = []

    # ======================================
    # PEAR
    # ======================================

    if (
        "a-line" in shape
        or "anarkali" in shape
        or "flared" in hem
    ):
        body_types.append("pear")

    # ======================================
    # APPLE
    # ======================================

    if (
        "straight" in shape
        or "v-neck" in neck
    ):
        body_types.append("apple")

    # ======================================
    # RECTANGLE
    # ======================================

    if (
        "straight fit" in fit
        or "regular fit" in fit
        or "solid" in pattern
    ):
        body_types.append("rectangle")

    # ======================================
    # INVERTED TRIANGLE
    # ======================================

    if (
        "boat neck" in neck
        or "off-shoulder" in neck
        or "shoulder straps" in neck
    ):
        body_types.append("inverted_triangle")

    # ======================================
    # HOURGLASS
    # ======================================

    if (
        "wrap" in outfit_type
        or "cinched" in outfit_type
        or "belt" in style_tip
    ):
        body_types.append("hourglass")

    # ======================================

    if len(body_types) == 0:
        return "neutral"

    return ",".join(sorted(set(body_types)))

df["body_type"] = df.apply(
    assign_body_type,
    axis=1
)

# ==========================================
# UNDERTONE LABELING
# ==========================================

warm_colors = [
    "yellow",
    "mustard",
    "orange",
    "gold",
    "olive",
    "peach",
    "maroon",
    "brown",
    "rust",
    "cream",
]

cool_colors = [
    "blue",
    "purple",
    "pink",
    "lavender",
    "grey",
    "gray",
    "black",
    "navy",
    "silver",
    "white",
]

neutral_colors = [
    "green",
    "red",
    "beige",
    "multicolor",
]

def assign_undertone(color):

    if pd.isna(color):
        return "neutral"

    color = str(color).lower()

    for c in warm_colors:

        if c in color:
            return "warm"

    for c in cool_colors:

        if c in color:
            return "cool"

    for c in neutral_colors:

        if c in color:
            return "neutral"

    return "neutral"

df["undertone"] = df["colour"].apply(
    assign_undertone
)

# ==========================================
# REMOVE EMPTY IMAGE ROWS
# ==========================================

df = df[df["img"].notna()]
df = df[df["img"] != ""]

# ==========================================
# ANALYTICS
# ==========================================

print("\n===== BODY TYPE COUNTS =====")
print(df["body_type"].value_counts())

print("\n===== UNDERTONE COUNTS =====")
print(df["undertone"].value_counts())

print("\n===== OCCASION COUNTS =====")
print(df["occasion_clean"].value_counts())

print("\n===== TOP SHAPE COUNTS =====")
print(df["top_shape"].value_counts())

print("\n===== COMBINED ANALYTICS =====")

combo = (
    df.groupby(
        ["occasion_clean", "undertone", "body_type"]
    )
    .size()
    .reset_index(name="count")
    .sort_values(by="count", ascending=False)
)

print(combo.head(50))

# ==========================================
# SAVE FINAL DATASET
# ==========================================

important_cols = [
    "p_id",
    "name",
    "brand",
    "img",
    "colour",
    "occasion_clean",
    "clean_description",
    "top_shape",
    "neck",
    "top_pattern",
    "top_fabric",
    "fit",
    "body_type",
    "undertone",
]

final_df = df[important_cols]

final_df.to_csv(
    "fashion_dataset_processed.csv",
    index=False
)

print("\nDataset saved successfully.")
print("\nSaved as: fashion_dataset_processed.csv")


===== DATASET COLUMNS =====
Index(['p_id', 'name', 'colour', 'brand', 'img', 'description',
       'occasion_clean', 'top_shape', 'top_length', 'top_pattern',
       'print_or_pattern_type', 'neck', 'sleeve_length', 'top_hemline',
       'main_trend', 'top_fabric', 'colour_family', 'fit', 'pattern_coverage',
       'occasion', 'type', 'waist_rise', 'body_shape_id', 'where_to_wear',
       'style_tip', 'fabric', 'pattern', 'undertone_match', 'silhouette',
       'visual_weight', 'body_balance', 'fabric_structure'],
      dtype='object')


Exception: No attribute column found! Expected 'attribute_dict' or 'p_attributes'

In [14]:
import pandas as pd
import re

# ==========================================
# LOAD DATASET
# ==========================================

df = pd.read_csv("fashion_ai_ready_dataset.csv")

print("\n===== DATASET COLUMNS =====")
print(df.columns)

# ==========================================
# CLEAN TEXT COLUMNS
# ==========================================

text_cols = [
    "top_shape",
    "top_length",
    "top_pattern",
    "print_or_pattern_type",
    "neck",
    "sleeve_length",
    "top_hemline",
    "main_trend",
    "top_fabric",
    "colour_family",
    "fit",
    "pattern_coverage",
    "occasion",
    "type",
    "waist_rise",
    "where_to_wear",
    "style_tip",
    "fabric",
    "pattern",
    "colour",
]

for col in text_cols:

    if col in df.columns:

        df[col] = (
            df[col]
            .astype(str)
            .str.lower()
            .str.strip()
            .replace("none", pd.NA)
            .replace("nan", pd.NA)
        )

# ==========================================
# CLEAN DESCRIPTION
# ==========================================

def clean_description(text):

    if pd.isna(text):
        return ""

    text = re.sub(r"<.*?>", " ", str(text))
    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["clean_description"] = df["description"].apply(
    clean_description
)

# ==========================================
# BODY TYPE LABELING
# ==========================================

def assign_body_type(row):

    shape = str(row["top_shape"])
    hem = str(row["top_hemline"])
    neck = str(row["neck"])
    fit = str(row["fit"])
    pattern = str(row["print_or_pattern_type"])
    outfit_type = str(row["type"])
    silhouette = str(row["silhouette"])

    body_types = []

    # PEAR
    if (
        "a-line" in shape
        or "anarkali" in shape
        or "flared" in hem
    ):
        body_types.append("pear")

    # APPLE
    if (
        "straight" in shape
        or "v-neck" in neck
    ):
        body_types.append("apple")

    # RECTANGLE
    if (
        "regular fit" in fit
        or "straight fit" in fit
        or "solid" in pattern
    ):
        body_types.append("rectangle")

    # INVERTED TRIANGLE
    if (
        "boat neck" in neck
        or "off-shoulder" in neck
        or "shoulder straps" in neck
    ):
        body_types.append("inverted_triangle")

    # HOURGLASS
    if (
        "wrap" in outfit_type
        or "balanced" in silhouette
    ):
        body_types.append("hourglass")

    if len(body_types) == 0:
        return "neutral"

    return ",".join(sorted(set(body_types)))

df["body_type"] = df.apply(
    assign_body_type,
    axis=1
)

# ==========================================
# UNDERTONE LABELING
# ==========================================

warm_colors = [
    "yellow",
    "mustard",
    "orange",
    "gold",
    "olive",
    "peach",
    "maroon",
    "brown",
    "rust",
    "cream",
]

cool_colors = [
    "blue",
    "purple",
    "pink",
    "lavender",
    "grey",
    "gray",
    "black",
    "navy",
    "silver",
    "white",
]

neutral_colors = [
    "green",
    "red",
    "beige",
    "multicolor",
]

def assign_undertone(color):

    if pd.isna(color):
        return "neutral"

    color = str(color).lower()

    for c in warm_colors:

        if c in color:
            return "warm"

    for c in cool_colors:

        if c in color:
            return "cool"

    for c in neutral_colors:

        if c in color:
            return "neutral"

    return "neutral"

df["undertone"] = df["colour"].apply(
    assign_undertone
)

# ==========================================
# REMOVE EMPTY IMAGE ROWS
# ==========================================

df = df[df["img"].notna()]
df = df[df["img"] != ""]

# ==========================================
# ANALYTICS
# ==========================================

print("\n===== BODY TYPE COUNTS =====")
print(df["body_type"].value_counts())

print("\n===== UNDERTONE COUNTS =====")
print(df["undertone"].value_counts())

print("\n===== OCCASION COUNTS =====")
print(df["occasion_clean"].value_counts())

print("\n===== COMBINED ANALYTICS =====")

combo = (
    df.groupby(
        ["occasion_clean", "undertone", "body_type"]
    )
    .size()
    .reset_index(name="count")
    .sort_values(by="count", ascending=False)
)

print(combo.head(50))

# ==========================================
# SAVE FINAL DATASET
# ==========================================

important_cols = [
    "p_id",
    "name",
    "brand",
    "img",
    "colour",
    "occasion_clean",
    "clean_description",
    "top_shape",
    "neck",
    "top_pattern",
    "top_fabric",
    "fit",
    "body_type",
    "undertone",
]

final_df = df[important_cols]

final_df.to_csv(
    "fashion_dataset_processed.csv",
    index=False
)

print("\nDataset saved successfully.")
print("\nSaved as: fashion_dataset_processed.csv")


===== DATASET COLUMNS =====
Index(['p_id', 'name', 'colour', 'brand', 'img', 'description',
       'occasion_clean', 'top_shape', 'top_length', 'top_pattern',
       'print_or_pattern_type', 'neck', 'sleeve_length', 'top_hemline',
       'main_trend', 'top_fabric', 'colour_family', 'fit', 'pattern_coverage',
       'occasion', 'type', 'waist_rise', 'body_shape_id', 'where_to_wear',
       'style_tip', 'fabric', 'pattern', 'undertone_match', 'silhouette',
       'visual_weight', 'body_balance', 'fabric_structure'],
      dtype='object')

===== BODY TYPE COUNTS =====
body_type
rectangle                            237
neutral                              164
apple                                 80
hourglass,pear                        34
pear                                  20
apple,rectangle                       20
apple,hourglass,pear                   8
inverted_triangle                      7
inverted_triangle,rectangle            5
apple,pear                             5
hourgla